In [ ]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

from nltk.corpus import stopwords
import nltk

In [ ]:
df = pd.read_csv("fr_dataset.csv")

# Colonnes attendues : message | label
texts = df["text"]
labels = df["labels"].map({"ham": 0, "spam": 1})

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"\d+", "", text)          # supprimer chiffres
    text = re.sub(r"[^\w\s]", "", text)      # supprimer ponctuation
    text = re.sub(r"\s+", " ", text).strip()
    return text



In [ ]:
texts = texts.apply(clean_text)

In [ ]:
nltk.download("stopwords")

french_stopwords = stopwords.words("french")

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words=french_stopwords,
    lowercase=True,
    ngram_range=(1, 2)
)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
X = vectorizer.fit_transform(texts)
y = labels.values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [ ]:
input_dim = X_train.shape[1]

model = Sequential()
model.add(Dense(128, activation="relu", input_shape=(input_dim,)))
model.add(Dropout(0.3))
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.3))
model.add(Dense(1, activation="sigmoid"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9999 - loss: 9.5669e-04 - val_accuracy: 0.9709 - val_loss: 0.1883
Epoch 2/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.9999 - loss: 0.0011 - val_accuracy: 0.9731 - val_loss: 0.1890
Epoch 3/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 1.0000 - loss: 3.7705e-04 - val_accuracy: 0.9742 - val_loss: 0.1980
Epoch 4/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.9999 - loss: 7.1422e-04 - val_accuracy: 0.9742 - val_loss: 0.1961
Epoch 5/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9999 - loss: 7.3154e-04 - val_accuracy: 0.9720 - val_loss: 0.1994
Epoch 6/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9994 - loss: 0.0023 - val_accuracy: 0.9709 - val_loss: 0.2070
Epoch 7/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9995 - loss: 0.0020 - val_accuracy: 0.9709 - val_loss: 0.2095
Epoch 8/15
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9999 - loss: 4

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob >= 0.5).astype(int)

print("\nAccuracy :", accuracy_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
print("\nRapport de classification :\n")
print(classification_report(y_test, y_pred))

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

Accuracy : 0.9820627802690582
F1-score : 0.9295774647887324

Rapport de classification :

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       966
           1       0.98      0.89      0.93       149

    accuracy                           0.98      1115
   macro avg       0.98      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [ ]:
def predict_message(message):
    message = clean_text(message)
    vector = vectorizer.transform([message]).toarray()
    prob = model.predict(vector)[0][0]

    label = "SPAM" if prob >= 0.5 else "HAM"
    return label, prob

In [ ]:
test_message = "appelle ce numéro"
label, confidence = predict_message(test_message)

print("\nMessage :", test_message)
print("Résultat :", label)
print("Confiance :", round(confidence * 100, 2), "%")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step

Message : appelle ce numéro
Résultat : SPAM
Confiance : 71.2 %


In [ ]:
import joblib
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')


['spam_model.h5']

In [ ]:
model.save("spam_model.h5")